# LLM-from-Scratch (100M Parameters) — Kaggle Edition

Train a 100M parameter GPT-style Transformer from scratch on Kaggle's T4 GPU.

**Session Chaining Workflow:**
1. Run this notebook (trains for ~2,500 steps / ~2–3 hours).
2. When it finishes, **Save Version** (commit) to preserve outputs.
3. Start a new session from that saved version: click the **Versions** tab → select the latest version → **Run**.
4. The new session auto-detects the previous checkpoint and resumes.

> **Tip:** Kaggle free T4 sessions last up to 12 hours. We default to 2,500 steps per session to stay well within the limit.

## 1. Verify GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q torch numpy tqdm requests matplotlib regex

import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Copy Code from Kaggle Dataset (or Clone)

> **Fast path:** Attach the `llm-from-scratch` code dataset as an Input.
> The notebook copies it from `/kaggle/input/` — no network, no git, instant.
> **Fallback:** If the dataset isn't attached, it does a shallow git clone.

In [ ]:
import os
import shutil

WORKING_DIR = "/kaggle/working/llm-from-scratch"
INPUT_CODE_DIR = "/kaggle/input/llm-from-scratch"  # Attach code as a Kaggle Dataset
REPO_URL = "https://github.com/avneeshjadhav04/llm-from-scratch.git"

os.makedirs(WORKING_DIR, exist_ok=True)

if os.path.isdir(INPUT_CODE_DIR):
    # Fast path: copy code from attached dataset (no network)
    print(f"Copying code from Kaggle Dataset: {INPUT_CODE_DIR}")
    for item in os.listdir(INPUT_CODE_DIR):
        src = os.path.join(INPUT_CODE_DIR, item)
        dst = os.path.join(WORKING_DIR, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print("Code copied successfully.")
elif os.path.isdir(os.path.join(WORKING_DIR, ".git")):
    # Chained session with existing repo — pull latest
    !cd {WORKING_DIR} && git pull
else:
    # Fallback: shallow clone from GitHub
    print("Kaggle Dataset not attached. Falling back to git clone...")
    !git clone --depth 1 {REPO_URL} {WORKING_DIR}

%cd {WORKING_DIR}

# Ensure output directories exist
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)

## 3. Auto-Resume: Detect Checkpoints from Previous Session

Kaggle mounts previous notebook outputs (when added as **Input**) under `/kaggle/input/`.
This cell scans all input directories and copies the latest checkpoint into the working tree.

In [ ]:
import glob
import shutil

INPUT_DIR = "/kaggle/input"
CHECKPOINT_DIR = "checkpoints"
LOG_DIR = "logs"

# Scan /kaggle/input/ and working dir for existing checkpoints
checkpoint_sources = [CHECKPOINT_DIR]
for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        if f.endswith(".pt"):
            checkpoint_sources.append(root)
            break

found_checkpoints = []
for src in set(checkpoint_sources):
    found_checkpoints.extend(glob.glob(f"{src}/*.pt"))

if found_checkpoints:
    # Pick the latest by step number in filename
    latest = max(
        found_checkpoints,
        key=lambda x: int(os.path.basename(x).split("_")[-1].replace(".pt", ""))
    )
    print(f"Found checkpoint: {latest}")
    if not latest.startswith(CHECKPOINT_DIR):
        shutil.copy(latest, CHECKPOINT_DIR)
        print(f"  -> Copied to {CHECKPOINT_DIR}")

    # Also pull in any log files so plots/CSV stay continuous
    for src in set(checkpoint_sources):
        for log_file in glob.glob(f"{src}/*training_log.csv"):
            if not log_file.startswith(LOG_DIR):
                shutil.copy(log_file, LOG_DIR)
                print(f"  -> Copied log: {os.path.basename(log_file)}")
else:
    print("No checkpoints found. Starting training from scratch.")

## 4. Prepare Data (Tokenize Corpus)

This downloads Wikitext-2 and trains the BPE tokenizer.
On a fresh session this takes ~5–10 minutes.

In [ ]:
!python prepare_data.py

## 5. Train the 100M Model

We limit each session to **2,500 steps** (~2–3 hours on T4) so we never hit the 12-hour wall.
If a checkpoint exists, training auto-resumes from the latest step.

In [ ]:
SESSION_STEPS = 2500
!python train.py --max_steps_per_session {SESSION_STEPS}

## 6. Verify Outputs

Check what was produced this session.
**Remember to Save Version (commit) so these outputs are available for the next session!**

In [ ]:
print("\nCheckpoints:")
for f in sorted(os.listdir("checkpoints")):
    size_mb = os.path.getsize(f"checkpoints/{f}") / (1024 * 1024)
    print(f"  {f} ({size_mb:.1f} MB)")

print("\nLogs:")
for f in sorted(os.listdir("logs")):
    print(f"  {f}")

# Show latest training metrics
import csv
log_files = glob.glob("logs/*training_log.csv")
if log_files:
    with open(log_files[0], "r") as f:
        rows = list(csv.DictReader(f))
        if rows:
            last = rows[-1]
            print(f"\nLatest metrics (step {last['step']}):")
            print(f"  Train Loss: {last['train_loss']}")
            print(f"  Val Loss:   {last.get('val_loss', 'N/A')}")
            print(f"  LR:         {last['lr']}")
            print(f"  Tok/s:      {last.get('tokens_per_sec', 'N/A')}")

## 7. Generate Text (Optional)

Quick sanity check using the latest checkpoint.

In [ ]:
import glob
ckpts = sorted(glob.glob("checkpoints/*.pt"))
if ckpts:
    latest_ckpt = ckpts[-1]
    !python generate.py \
        --checkpoint {latest_ckpt} \
        --prompt "The future of artificial intelligence is" \
        --temperature 0.8 \
        --top_k 50 \
        --max_new_tokens 128 \
        --device cuda
else:
    print("No checkpoint available yet. Train first.")